In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM

processor = AutoProcessor.from_pretrained("google/functiongemma-270m-it", device_map="auto")
model = AutoModelForCausalLM.from_pretrained("google/functiongemma-270m-it", dtype="auto", device_map="auto")


/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 624.00it/s]


In [3]:
weather_function_schema = {
    "type": "function",
    "function": {
        "name": "get_current_temperature",
        "description": "Gets the current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, e.g. San Francisco",
                },
            },
            "required": ["location"],
        },
    }
}

message = [
    # ESSENTIAL SYSTEM PROMPT:
    # This line activates the model's function calling logic.
    {
        "role": "developer",
        "content": "You are a model that can do function calling with the following functions"
    },
    # {
    #     "role": "user", 
    #     "content": "What's the temperature in London?"
    # },
    {
        "role": "user", 
        "content": "What's the stock price of Google?"
    }
]

inputs = processor.apply_chat_template(message, tools=[weather_function_schema], add_generation_prompt=True, return_dict=True, return_tensors="pt")

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print(output)

I cannot provide real-time stock price data. My available tools are focused on temperature and location-based requests. I cannot access or analyze financial market information.


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

gemma_270m = 'google/gemma-3-270m'
gemma_2b = 'google/gemma-2b-it'
tokenizer = AutoTokenizer.from_pretrained(gemma_270m)
model = AutoModelForCausalLM.from_pretrained(gemma_270m, device_map="cpu")

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 7476.36it/s]


In [4]:
input_text = """
<bos>SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_tip",
    "description": "Calculate the tip amount for a bill",
    "parameters": {
        "type": "object",
        "properties": {
            "bill_amount": {
                "type": "number",
                "description": "The total bill amount"
            },
            "tip_percentage": {
                "type": "number",
                "description": "The tip percentage"
            }
        },
        "required": [
            "bill_amount",
            "tip_percentage"
        ]
    }
}


USER: Hi, I need help with calculating a tip. My bill is $50 and I want to leave a 20% tip. 
ASSISTANT:
"""
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**input_ids, generation_config=GenerationConfig.from_dict({"max_new_tokens": 1000})).to('cpu')
print(tokenizer.decode(outputs[0]))

<bos>
<bos>SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_tip",
    "description": "Calculate the tip amount for a bill",
    "parameters": {
        "type": "object",
        "properties": {
            "bill_amount": {
                "type": "number",
                "description": "The total bill amount"
            },
            "tip_percentage": {
                "type": "number",
                "description": "The tip percentage"
            }
        },
        "required": [
            "bill_amount",
            "tip_percentage"
        ]
    }
}


USER: Hi, I need help with calculating a tip. My bill is $50 and I want to leave a 20% tip. 
ASSISTANT:
{
    "name": "calculate_tip_amount",
    "description": "Calculate the tip amount for a bill",
    "parameters": {
        "type": "object",
        "properties": {
            "bill_amount": {
                "type": "number",
                "descrip